# Capítulo 5 · Actividad: Titanic, ¿árbol o vecinos?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riosinda/ml-course/blob/main/notebooks/exercises/chapter_05/04_actividad_titanic_arbol_knn.ipynb)

Compara un **árbol de decisión** y **KNN** para predecir si un pasajero del Titanic sobrevivió. Primero entiende los datos; el preprocesamiento lo decides tú. Los dos modelos tienen que evaluarse sobre la misma división de train y test, y con las mismas particiones de validación cruzada.

Cada fila es un pasajero. Para esta actividad usa las siguientes variables:

| Variable | Significado |
|---|---|
| `survived` | **Target**: `0` = no sobrevivió, `1` = sobrevivió |
| `pclass` | Clase del boleto: primera, segunda o tercera |
| `sex` | Sexo del pasajero |
| `age` | Edad en años |
| `sibsp` | Número de hermanos y cónyuges a bordo |
| `parch` | Número de padres e hijos a bordo |
| `fare` | Tarifa pagada |
| `embarked` | Puerto de embarque: `C`, `Q` o `S` |

Hay valores faltantes en `age` y `embarked`. Las clases de supervivencia no están balanceadas y las variables numéricas tienen escalas distintas.

**Usa únicamente las siete variables predictoras de la tabla.** El CSV contiene otras columnas: `alive`, por ejemplo, expresa la misma respuesta que `survived`. Incluirla entre las entradas sería una fuga de información: el modelo recibiría la respuesta que debe predecir. Tampoco incluyas `survived` entre las entradas.

Como mínimo, tu notebook debe incluir:

- La exploración de antes de modelar: forma de la tabla, nulos, balance de supervivencia, distribuciones y relación de las variables con la supervivencia.
- Una división estratificada de train y test y validación cruzada estratificada sobre train. Usa `random_state=42` en la división, en el árbol y al mezclar las particiones de validación.
- Un **árbol de decisión** y un **KNN**, los dos para clasificar. Decide cómo tratar los nulos, cómo codificar las variables categóricas y si estandarizas las numéricas: a KNN la escala le importa, al árbol no. Integra el preprocesamiento en un `Pipeline` para que se ajuste solo con los datos de entrenamiento de cada partición.
- La profundidad del árbol (y `min_samples_leaf`, si quieres) elegida con `GridSearchCV` sobre la validación cruzada.
- Para KNN, varios valores de `k`, incluyendo `k=1`, una **gráfica de la accuracy de train y de validación contra `k`** y el `k` que alcanza el máximo en validación. Distingue la accuracy sobre los mismos datos usados para entrenar de la obtenida en las particiones de validación. El test se usa una vez, al final, para evaluar ambos modelos elegidos, no para escoger hiperparámetros.
- Una **tabla** con los dos modelos: accuracy en validación cruzada (media y desviación estándar) y en prueba, y la matriz de confusión de cada uno en prueba. Compara también con una referencia sencilla: predecir siempre la clase mayoritaria de train.
- Tus **conclusiones**: con cuál te quedas y por qué. Explica la diferencia entre train y validación con `k=1`, cómo cambia al aumentar `k` y qué errores comete cada modelo al predecir supervivencia.

No hay una accuracy objetivo ni un ganador obligatorio. El objetivo es justificar la comparación con los resultados: `k=1` puede generalizar bien y la curva de validación no tiene por qué mejorar siempre al aumentar `k`. Si los modelos obtienen resultados muy parecidos, reconócelo en tus conclusiones.

1. **Copia esta notebook a `workspace/`** y trabaja sobre la copia.
2. Si todavía no tienes los datos: `uv run python scripts/download_data.py`


In [ ]:
from pathlib import Path

import pandas as pd

raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
df = pd.read_csv(raiz / "data" / "titanic.csv")
df.head()


## Exploración inicial

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

sns.set_theme(style="whitegrid")

# Solo las 7 variables predictoras permitidas (sin `alive` ni `survived`: serían fuga de información)
numeric_features = ["age", "fare", "sibsp", "parch"]
categorical_features = ["pclass", "sex", "embarked"]
features = numeric_features + categorical_features
df = df[features + ["survived"]]

print("Forma:", df.shape)
df.isna().sum()

In [ ]:
print(df["survived"].value_counts(normalize=True).round(3))
sns.countplot(data=df, x="survived")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for col, ax in zip(["sex", "pclass", "embarked"], axes):
    sns.barplot(data=df, x=col, y="survived", ax=ax)
    ax.set_ylabel("proporción de supervivientes")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for col, ax in zip(numeric_features, axes):
    sns.histplot(data=df, x=col, hue="survived", bins=20, ax=ax, legend=(col == "age"))
plt.tight_layout()
plt.show()

Hallazgos de la exploración: faltan 177 edades y 2 puertos de embarque; solo el 38 % sobrevivió (clases desbalanceadas); las mujeres y los pasajeros de primera clase sobrevivieron mucho más; `fare` y `age` tienen escalas muy distintas y `fare` tiene una cola larga.

## División en entrenamiento y prueba

Estratificada, para conservar la proporción de supervivientes. Las **mismas** particiones (train/test y los 5 folds) se usan para los dos modelos.

In [ ]:
X = df[features]
y = df["survived"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(X_train.shape, X_test.shape)

## Preprocesamiento en un `Pipeline`

- **Nulos:** `age` con la mediana; `embarked` con la moda (el valor más frecuente).
- **Categóricas:** `OneHotEncoder`.
- **Escala:** el árbol no la necesita; KNN sí, porque calcula distancias y `fare` (hasta 500) dominaría sobre `sibsp` (0 a 8). Por eso KNN lleva `StandardScaler`.

Todo va dentro del `Pipeline`, así los valores de imputación y la escala se calculan solo con los datos de entrenamiento de cada fold.

In [ ]:
def make_preprocessor(scale):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        numeric_steps.append(("scaler", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(numeric_steps), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_features),
    ])

tree_pipe = Pipeline([
    ("prep", make_preprocessor(scale=False)),
    ("model", DecisionTreeClassifier(random_state=42)),
])
knn_pipe = Pipeline([
    ("prep", make_preprocessor(scale=True)),
    ("model", KNeighborsClassifier()),
])

## Árbol de decisión: profundidad con `GridSearchCV`

Un árbol sin límite memoriza el train (sobreajuste). Buscamos `max_depth` y `min_samples_leaf` con validación cruzada.

In [ ]:
tree_grid = GridSearchCV(
    tree_pipe,
    param_grid={
        "model__max_depth": [1, 2, 3, 4, 5, 6, 8, 10, None],
        "model__min_samples_leaf": [1, 5, 10, 20],
    },
    scoring="accuracy", cv=cv,
)
tree_grid.fit(X_train, y_train)
print("Mejores parámetros:", tree_grid.best_params_)
print(f"Accuracy en validación: {tree_grid.best_score_:.4f}")
best_tree = tree_grid.best_estimator_

## KNN: varios valores de `k`

Para cada `k` calculamos la accuracy en los datos con que se entrena (train) y en los folds de validación. Con `k=1` cada punto de train es su propio vecino más cercano, así que la accuracy en train es casi perfecta aunque no signifique nada: lo que importa es la validación.

In [ ]:
k_values = list(range(1, 51))
rows = []
for k in k_values:
    model = knn_pipe.set_params(model__n_neighbors=k)
    s = cross_validate(model, X_train, y_train, cv=cv, scoring="accuracy", return_train_score=True)
    rows.append({"k": k, "train": s["train_score"].mean(), "validacion": s["test_score"].mean(),
                 "validacion_std": s["test_score"].std()})
knn_curve = pd.DataFrame(rows).set_index("k")

best_k = int(knn_curve["validacion"].idxmax())
print("k = 1  ->", knn_curve.loc[1, ["train", "validacion"]].round(4).to_dict())
print(f"Mejor k: {best_k} (accuracy en validación = {knn_curve.loc[best_k, 'validacion']:.4f})")

plt.figure(figsize=(8, 4))
plt.plot(knn_curve.index, knn_curve["train"], label="train")
plt.plot(knn_curve.index, knn_curve["validacion"], label="validación")
plt.axvline(best_k, color="gray", linestyle="--", label=f"mejor k = {best_k}")
plt.xlabel("k (número de vecinos)")
plt.ylabel("accuracy")
plt.legend()
plt.show()

## Comparación final

El test se usa **una sola vez**, para los dos modelos ya elegidos, más una referencia que siempre predice la clase mayoritaria de train.

In [ ]:
best_knn = knn_pipe.set_params(model__n_neighbors=best_k)
dummy = DummyClassifier(strategy="most_frequent")

final_models = {"Referencia (mayoritaria)": dummy, "Árbol": best_tree, f"KNN (k={best_k})": best_knn}
rows, predictions = [], {}
for name, model in final_models.items():
    cv_scores = cross_validate(model, X_train, y_train, cv=cv, scoring="accuracy")["test_score"]
    model.fit(X_train, y_train)
    predictions[name] = model.predict(X_test)
    rows.append({
        "modelo": name,
        "accuracy_cv_media": cv_scores.mean(),
        "accuracy_cv_std": cv_scores.std(),
        "accuracy_test": accuracy_score(y_test, predictions[name]),
    })
pd.DataFrame(rows).set_index("modelo").round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for (name, pred), ax in zip([(n, p) for n, p in predictions.items() if "Referencia" not in n], axes):
    ConfusionMatrixDisplay.from_predictions(
        y_test, pred, display_labels=["no sobrevivió", "sobrevivió"], cmap="Blues", ax=ax, colorbar=False
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()

## Conclusiones

- **Los datos:** 891 pasajeros, 38 % sobrevivió, así que la referencia de "siempre decir que no sobrevivió" ya acierta ~61 % (0.615 en prueba). Los dos modelos la superan por bastante (~0.79 en prueba), es decir, sí aprendieron algo útil.
- **Resultados:** en validación cruzada el KNN (k=23) obtuvo 0.824 ± 0.023 y el árbol (`max_depth=4`) 0.813 ± 0.013. En prueba **empatan: 0.788 los dos**. La diferencia en validación (0.011) es menor que la desviación entre folds, así que no hay un ganador claro; con 179 pasajeros de prueba, un acierto de más o de menos cambia ~0.6 puntos.
- **`k=1`: train vs. validación.** Con `k=1` la accuracy en train es 0.978 y en validación 0.757. En train cada pasajero es su propio vecino más cercano, así que "se acuerda" de su respuesta: es memorización (sobreajuste), no capacidad de predecir. Lo honesto es la validación, y ahí `k=1` es de los peores valores porque cada predicción depende de un solo vecino, que puede ser ruido.
- **Al aumentar `k`:** la accuracy en train baja (0.978 → 0.855 con k=5 → 0.832 con k=23) y la de validación sube hasta una meseta (~0.82 entre k=17 y k=30; el máximo, 0.8244, está en k=23 y es prácticamente igual que en k=18 y k=26). Con k=50 ya baja otra vez (0.781): demasiados vecinos suavizan tanto que el modelo se vuelve demasiado simple (subajuste). La curva de validación no mejora siempre al subir `k`: hay un punto medio.
- **Errores en prueba** (de 69 supervivientes reales y 110 fallecidos):
  - Árbol: 31 supervivientes mal clasificados como fallecidos y 7 fallecidos como supervivientes. Acierta mucho al decir "sobrevive" (38 de 45) pero se le escapa casi la mitad de los que sobrevivieron (recall 0.55).
  - KNN: 25 supervivientes perdidos y 13 falsas alarmas. Detecta más supervivientes (recall 0.64) a costa de más falsas alarmas (precisión 0.77 vs. 0.84 del árbol).
  - Los dos fallan sobre todo en el mismo sentido: les cuesta reconocer a los supervivientes que no encajan en el patrón "mujer o primera clase".
- **¿Con cuál me quedo?** Con el **árbol**, porque con la misma accuracy en prueba es más simple de explicar (se pueden leer sus reglas: sexo, clase, edad), no necesita escalar y su validación es más estable (menor desviación). Si me importara más encontrar a los supervivientes que evitar falsas alarmas, elegiría KNN por su mayor recall. Hay que reconocer que, con estos datos, la diferencia entre ambos es pequeña.